# NHL Data Pipeline — Game & Season Stats

In [ ]:
import requests
import sqlite3
import pandas as pd
import time

DB_PATH = "../data/nhl.db"
BOXSCORE_URL = "https://api-web.nhle.com/v1/gamecenter/{game_id}/boxscore"
PLAYER_URL = "https://api-web.nhle.com/v1/player/{player_id}/landing"
TIMEOUT = 15
SLEEP = 0.3
COMMIT_EVERY = 25

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

## Game stats (skaters, per game)

In [ ]:
def parse_boxscore(box, game_id, home_id, away_id):
    rows = []
    stats = box.get("playerByGameStats", {})
    for side, team_id in (("homeTeam", home_id), ("awayTeam", away_id)):
        side_stats = stats.get(side, {})
        for group in ("forwards", "defense"):
            for p in side_stats.get(group, []):
                player_id = p.get("playerId")
                if player_id is None:
                    continue
                rows.append({
                    "game_id": game_id,
                    "player_id": player_id,
                    "team_id": team_id,
                    "goals": p.get("goals"),
                    "assists": p.get("assists"),
                    "points": p.get("points"),
                    "shots_on_goal": p.get("sog", p.get("shots")),
                    "penalty_min": p.get("pim"),
                    "toi": p.get("toi"),
                    "plus_minus": p.get("plusMinus"),
                })
    return rows

In [ ]:
games = pd.read_sql(
    "SELECT game_id, home_team_id, away_team_id FROM games WHERE game_state = 'OFF' ORDER BY game_date",
    conn,
)
if games.empty:
    raise ValueError("No completed games found — check notebook 03 ran with a completed SEASON.")

failed_games, total_rows = [], 0

for i, row in enumerate(games.itertuples(index=False), 1):
    try:
        resp = requests.get(BOXSCORE_URL.format(game_id=row.game_id), timeout=TIMEOUT)
        resp.raise_for_status()
        parsed = parse_boxscore(resp.json(), row.game_id, row.home_team_id, row.away_team_id)

        cur.execute("DELETE FROM game_stats WHERE game_id = ?", (row.game_id,))
        for r in parsed:
            cur.execute(
                "INSERT INTO game_stats (game_id, player_id, team_id, goals, assists, points, "
                "shots_on_goal, penalty_min, toi, plus_minus) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                (r["game_id"], r["player_id"], r["team_id"], r["goals"], r["assists"], r["points"],
                 r["shots_on_goal"], r["penalty_min"], r["toi"], r["plus_minus"]),
            )
        total_rows += len(parsed)
    except Exception as e:
        failed_games.append((row.game_id, str(e)))

    if i % COMMIT_EVERY == 0:
        conn.commit()
        print(f"{i}/{len(games)} games, {total_rows} rows")

    time.sleep(SLEEP)

conn.commit()
print(f"game_stats: {total_rows} rows loaded, {len(failed_games)} games failed")

## Season stats (skaters and goalies)

In [ ]:
def parse_skater(data, player_id, team_id):
    featured = data.get("featuredStats")
    if not featured:
        return None
    season = featured.get("season")
    sub = featured.get("regularSeason", {}).get("subSeason")
    if not sub:
        return None
    avg_toi = next(
        (e.get("avgToi") for e in data.get("seasonTotals", [])
         if e.get("season") == season and e.get("leagueAbbrev") == "NHL" and e.get("gameTypeId") == 2),
        None,
    )
    return {
        "player_id": player_id,
        "season": str(season),
        "team_id": team_id,
        "games_played": sub.get("gamesPlayed"),
        "goals": sub.get("goals"),
        "assists": sub.get("assists"),
        "points": sub.get("points"),
        "plus_minus": sub.get("plusMinus"),
        "penalty_min": sub.get("pim"),
        "shots": sub.get("shots"),
        "avg_toi": avg_toi,
    }


def parse_goalie(data, player_id, team_id):
    featured = data.get("featuredStats")
    if not featured:
        return None
    season = featured.get("season")
    sub = featured.get("regularSeason", {}).get("subSeason")
    if not sub:
        return None
    return {
        "player_id": player_id,
        "season": str(season),
        "team_id": team_id,
        "games_played": sub.get("gamesPlayed"),
        "wins": sub.get("wins"),
        "losses": sub.get("losses"),
        "ot_losses": sub.get("otLosses"),
        "save_pct": sub.get("savePctg"),
        "goals_against_avg": sub.get("goalsAgainstAvg"),
        "shutouts": sub.get("shutouts"),
        "saves": sub.get("saves"),
    }

In [ ]:
players = pd.read_sql("SELECT player_id, team_id, position FROM players ORDER BY player_id", conn)
failed_players, skater_rows, goalie_rows = [], 0, 0

for i, row in enumerate(players.itertuples(index=False), 1):
    try:
        resp = requests.get(PLAYER_URL.format(player_id=row.player_id), timeout=TIMEOUT)
        resp.raise_for_status()
        data = resp.json()

        if data.get("position") == "G":
            parsed = parse_goalie(data, row.player_id, row.team_id)
            if parsed:
                cur.execute(
                    "DELETE FROM goalie_season_stats WHERE player_id = ? AND season = ?",
                    (row.player_id, parsed["season"]),
                )
                cur.execute(
                    "INSERT INTO goalie_season_stats (player_id, season, team_id, games_played, wins, "
                    "losses, ot_losses, save_pct, goals_against_avg, shutouts, saves) "
                    "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                    (parsed["player_id"], parsed["season"], parsed["team_id"], parsed["games_played"],
                     parsed["wins"], parsed["losses"], parsed["ot_losses"], parsed["save_pct"],
                     parsed["goals_against_avg"], parsed["shutouts"], parsed["saves"]),
                )
                goalie_rows += 1
        else:
            parsed = parse_skater(data, row.player_id, row.team_id)
            if parsed:
                cur.execute(
                    "DELETE FROM skater_season_stats WHERE player_id = ? AND season = ?",
                    (row.player_id, parsed["season"]),
                )
                cur.execute(
                    "INSERT INTO skater_season_stats (player_id, season, team_id, games_played, goals, "
                    "assists, points, plus_minus, penalty_min, shots, avg_toi) "
                    "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                    (parsed["player_id"], parsed["season"], parsed["team_id"], parsed["games_played"],
                     parsed["goals"], parsed["assists"], parsed["points"], parsed["plus_minus"],
                     parsed["penalty_min"], parsed["shots"], parsed["avg_toi"]),
                )
                skater_rows += 1
    except Exception as e:
        failed_players.append((row.player_id, str(e)))

    if i % COMMIT_EVERY == 0:
        conn.commit()
        print(f"{i}/{len(players)} players, {skater_rows} skater rows, {goalie_rows} goalie rows")

    time.sleep(SLEEP)

conn.commit()
conn.close()
print(f"skater_season_stats: {skater_rows}, goalie_season_stats: {goalie_rows}, "
      f"{len(failed_players)} players failed")